# EDA — Previous Application & Behavioural Tables

Reference: PLAN_v2.md §2.3–§2.6.

**Tables**: `previous_application`, `installments_payments`, `POS_CASH_balance`, `credit_card_balance`.

**Checks performed:**
1. Sizes and join keys.
2. `NAME_CONTRACT_STATUS` distribution (drives Approved/Refused stratification).
3. `DAYS_*` sentinels in `previous_application` (also `365243`).
4. `installments_payments`: distribution of DPD (days past due).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import read_raw

sns.set_style('whitegrid')

In [ ]:
prev = read_raw('previous_application')
inst = read_raw('installments_payments')
pos = read_raw('pos_cash_balance')
cc = read_raw('credit_card_balance')

for name, df in [('previous_application', prev), ('installments_payments', inst), ('pos_cash_balance', pos), ('credit_card_balance', cc)]:
    print(f'{name:30s}  shape={df.shape}')

## 1. NAME_CONTRACT_STATUS in previous_application

In [ ]:
prev['NAME_CONTRACT_STATUS'].value_counts().sort('count', descending=True)

**Refused / Approved aggregations are highly predictive** — the `Refused` subset alone often produces top-importance features (PLAN §2.3).

## 2. DAYS_* sentinels in previous_application

In [ ]:
for c in ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
    n = (prev[c] == 365243).sum()
    print(f'{c}: {n:,} rows == 365243  ({n/prev.height:.2%})')

## 3. installments_payments — DPD distribution

`DPD = max(0, DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)` — days late.

In [ ]:
inst_with_dpd = inst.with_columns(
    pl.max_horizontal([pl.col('DAYS_ENTRY_PAYMENT') - pl.col('DAYS_INSTALMENT'), pl.lit(0)]).alias('DPD'),
)
dpd = inst_with_dpd['DPD'].drop_nulls()
print(f'DPD>0 rows: {(dpd > 0).sum():,}  ({(dpd > 0).mean():.2%})')
print(f'DPD percentiles: p50={dpd.quantile(0.5):.0f}  p95={dpd.quantile(0.95):.0f}  p99={dpd.quantile(0.99):.0f}  max={dpd.max():.0f}')